In [2]:
!pip install ray[tune]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 MB 11.7 MB/s eta 0:00:00


In [3]:
import os
import torch
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
import pandas as pd
from PIL import Image
import requests
import zipfile

import csv
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.optim import AdamW, SGD
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
from sklearn.metrics import precision_recall_fscore_support

import matplotlib.pyplot as plt

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.optuna import OptunaSearch
from functools import partial

In [ ]:
def download_tiny_imagenet():
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    data_dir = "/content/tiny-imagenet-200"
    if not os.path.exists(data_dir):
        print("Downloading Tiny ImageNet dataset...")
        response = requests.get(url, stream=True)
        with open("tiny-imagenet-200.zip", "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print("Extracting dataset...")
        with zipfile.ZipFile("tiny-imagenet-200.zip", "r") as zip_ref:
            zip_ref.extractall("/content")
        os.remove("tiny-imagenet-200.zip")
    return data_dir

# Define paths
data_dir = download_tiny_imagenet()
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val', 'images')
val_annotations = os.path.join(data_dir, 'val', 'val_annotations.txt')

print(f"Checking dataset directory: {data_dir}")
print(f"Directory exists: {os.path.exists(data_dir)}")
if os.path.exists(data_dir):
    print(f"Contents: {os.listdir(data_dir)}")
if not os.path.exists(train_dir):
    raise FileNotFoundError(f"Training directory not found: {train_dir}. Please verify the path.")
if not os.path.exists(val_dir):
    raise FileNotFoundError(f"Validation directory not found: {val_dir}. Please verify the path.")
if not os.path.exists(val_annotations):
    raise FileNotFoundError(f"Validation annotations not found: {val_annotations}. Please verify the path.")


class_folders = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
print(f"Found {len(class_folders)} class folders in {train_dir}. Expected 200 for Tiny ImageNet.")


Extracting dataset...
Checking dataset directory: /content/tiny-imagenet-200
Directory exists: True
Contents: ['wnids.txt', 'val', 'train', 'words.txt', 'test']
Found 200 class folders in /content/tiny-imagenet-200/train. Expected 200 for Tiny ImageNet.


In [1]:
!pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstall

In [ ]:
import os
import requests
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
import numpy as np
from PIL import Image
from torchmetrics import Accuracy, Precision, Recall, F1Score


def download_tiny_imagenet():
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    data_dir = "/content/tiny-imagenet-200"
    if not os.path.exists(data_dir):
        print("Downloading Tiny ImageNet dataset...")
        response = requests.get(url, stream=True)
        with open("tiny-imagenet-200.zip", "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print("Extracting dataset...")
        with zipfile.ZipFile("tiny-imagenet-200.zip", "r") as zip_ref:
            zip_ref.extractall("/content")
        os.remove("tiny-imagenet-200.zip")
    return data_dir


data_dir = download_tiny_imagenet()
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val', 'images')
val_annotations = os.path.join(data_dir, 'val', 'val_annotations.txt')

print(f"Checking dataset directory: {data_dir}")
print(f"Directory exists: {os.path.exists(data_dir)}")
if os.path.exists(data_dir):
    print(f"Contents: {os.listdir(data_dir)}")
if not os.path.exists(train_dir):
    raise FileNotFoundError(f"Training directory not found: {train_dir}. Please verify the path.")
if not os.path.exists(val_dir):
    raise FileNotFoundError(f"Validation directory not found: {val_dir}. Please verify the path.")
if not os.path.exists(val_annotations):
    raise FileNotFoundError(f"Validation annotations not found: {val_annotations}. Please verify the path.")


class_folders = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
print(f"Found {len(class_folders)} class folders in {train_dir}. Expected 200 for Tiny ImageNet.")


train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = ImageFolder(train_dir, transform=train_transforms)


class TinyImageNetValDataset(Dataset):
    def __init__(self, val_dir, val_annotations, transform=None):
        self.val_dir = val_dir
        self.transform = transform
       
        self.annotations = []
        with open(val_annotations, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                self.annotations.append((parts[0], parts[1])) 

        
        self.class_to_idx = {class_id: idx for idx, class_id in enumerate(sorted(set(ann[1] for ann in self.annotations)))}
        self.images = [(os.path.join(val_dir, ann[0]), self.class_to_idx[ann[1]]) for ann in self.annotations]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


val_dataset = TinyImageNetValDataset(val_dir, val_annotations, transform=val_transforms)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


model = models.mobilenet_v2(pretrained=True)


for param in model.features[:7].parameters():
    param.requires_grad = False


model.classifier[1] = nn.Linear(model.classifier[1].in_features, 200)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


top1_acc = Accuracy(task="multiclass", num_classes=200, top_k=1).to(device)
top5_acc = Accuracy(task="multiclass", num_classes=200, top_k=5).to(device)
precision = Precision(task="multiclass", num_classes=200, average='macro').to(device)
recall = Recall(task="multiclass", num_classes=200, average='macro').to(device)
f1 = F1Score(task="multiclass", num_classes=200, average='macro').to(device)


class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

Checking dataset directory: /content/tiny-imagenet-200
Directory exists: True
Contents: ['wnids.txt', 'val', 'train', 'words.txt', 'test']
Found 200 class folders in /content/tiny-imagenet-200/train. Expected 200 for Tiny ImageNet.


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:

num_epochs = 20
early_stopping = EarlyStopping(patience=7)

for epoch in range(num_epochs):
    
    model.train()
    running_loss = 0.0
    top1_acc.reset()
    top5_acc.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        top1_acc.update(outputs, labels)
        top5_acc.update(outputs, labels)
        precision.update(outputs, labels)
        recall.update(outputs, labels)
        f1.update(outputs, labels)

    train_loss = running_loss / len(train_loader)
    train_top1 = top1_acc.compute()
    train_top5 = top5_acc.compute()
    train_precision = precision.compute()
    train_recall = recall.compute()
    train_f1 = f1.compute()

    
    model.eval()
    val_loss = 0.0
    top1_acc.reset()
    top5_acc.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            top1_acc.update(outputs, labels)
            top5_acc.update(outputs, labels)
            precision.update(outputs, labels)
            recall.update(outputs, labels)
            f1.update(outputs, labels)

    val_loss = val_loss / len(val_loader)
    val_top1 = top1_acc.compute()
    val_top5 = top5_acc.compute()
    val_precision = precision.compute()
    val_recall = recall.compute()
    val_f1 = f1.compute()

    
    print(f'Epoch {epoch+1}:')
    print(f'Train - Loss: {train_loss:.4f}, TOP1 Acc: {train_top1:.4f}, TOP5 Acc: {train_top5:.4f}, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}')
    print(f'Val   - Loss: {val_loss:.4f}, TOP1 Acc: {val_top1:.4f}, TOP5 Acc: {val_top5:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}')

    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break

Epoch 1:
Train - Loss: 4.0902, TOP1 Acc: 0.1169, TOP5 Acc: 0.3155, Precision: 0.1003, Recall: 0.1169, F1: 0.1038
Val   - Loss: 3.2172, TOP1 Acc: 0.2388, TOP5 Acc: 0.5239, Precision: 0.2436, Recall: 0.2388, F1: 0.2045
Epoch 2:
Train - Loss: 3.5427, TOP1 Acc: 0.2024, TOP5 Acc: 0.4554, Precision: 0.1842, Recall: 0.2024, F1: 0.1880
Val   - Loss: 2.9058, TOP1 Acc: 0.3128, TOP5 Acc: 0.5971, Precision: 0.3321, Recall: 0.3128, F1: 0.2875
Epoch 3:
Train - Loss: 3.2944, TOP1 Acc: 0.2487, TOP5 Acc: 0.5123, Precision: 0.2319, Recall: 0.2487, F1: 0.2353
Val   - Loss: 2.7246, TOP1 Acc: 0.3386, TOP5 Acc: 0.6398, Precision: 0.3880, Recall: 0.3386, F1: 0.3237
Epoch 4:
Train - Loss: 3.1286, TOP1 Acc: 0.2826, TOP5 Acc: 0.5487, Precision: 0.2673, Recall: 0.2826, F1: 0.2707
Val   - Loss: 2.5228, TOP1 Acc: 0.3857, TOP5 Acc: 0.6801, Precision: 0.4096, Recall: 0.3857, F1: 0.3668
Epoch 5:
Train - Loss: 3.0167, TOP1 Acc: 0.3061, TOP5 Acc: 0.5720, Precision: 0.2915, Recall: 0.3061, F1: 0.2947
Val   - Loss: 2.410

In [ ]:
import os
import requests
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.datasets import ImageFolder
import numpy as np
from PIL import Image
from torchmetrics import Accuracy, Precision, Recall, F1Score


def download_tiny_imagenet():
    url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
    data_dir = "/content/tiny-imagenet-200"
    if not os.path.exists(data_dir):
        print("Downloading Tiny ImageNet dataset...")
        response = requests.get(url, stream=True)
        with open("tiny-imagenet-200.zip", "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
        print("Extracting dataset...")
        with zipfile.ZipFile("tiny-imagenet-200.zip", "r") as zip_ref:
            zip_ref.extractall("/content")
        os.remove("tiny-imagenet-200.zip")
    return data_dir


data_dir = download_tiny_imagenet()
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val', 'images')
val_annotations = os.path.join(data_dir, 'val', 'val_annotations.txt')

print(f"Checking dataset directory: {data_dir}")
print(f"Directory exists: {os.path.exists(data_dir)}")
if os.path.exists(data_dir):
    print(f"Contents: {os.listdir(data_dir)}")
if not os.path.exists(train_dir):
    raise FileNotFoundError(f"Training directory not found: {train_dir}. Please verify the path.")
if not os.path.exists(val_dir):
    raise FileNotFoundError(f"Validation directory not found: {val_dir}. Please verify the path.")
if not os.path.exists(val_annotations):
    raise FileNotFoundError(f"Validation annotations not found: {val_annotations}. Please verify the path.")


class_folders = [d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
print(f"Found {len(class_folders)} class folders in {train_dir}. Expected 200 for Tiny ImageNet.")


train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train_dataset = ImageFolder(train_dir, transform=train_transforms)


class TinyImageNetValDataset(Dataset):
    def __init__(self, val_dir, val_annotations, transform=None):
        self.val_dir = val_dir
        self.transform = transform
        
        self.annotations = []
        with open(val_annotations, 'r') as f:
            for line in f:
                parts = line.strip().split('\t')
                self.annotations.append((parts[0], parts[1]))  

        
        self.class_to_idx = {class_id: idx for idx, class_id in enumerate(sorted(set(ann[1] for ann in self.annotations)))}
        self.images = [(os.path.join(val_dir, ann[0]), self.class_to_idx[ann[1]]) for ann in self.annotations]

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label


val_dataset = TinyImageNetValDataset(val_dir, val_annotations, transform=val_transforms)


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


model = models.mobilenet_v2(pretrained=True)


model.classifier[1] = nn.Linear(model.classifier[1].in_features, 200)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


top1_acc = Accuracy(task="multiclass", num_classes=200, top_k=1).to(device)
top5_acc = Accuracy(task="multiclass", num_classes=200, top_k=5).to(device)
precision = Precision(task="multiclass", num_classes=200, average='macro').to(device)
recall = Recall(task="multiclass", num_classes=200, average='macro').to(device)
f1 = F1Score(task="multiclass", num_classes=200, average='macro').to(device)


class EarlyStopping:
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

Checking dataset directory: /content/tiny-imagenet-200
Directory exists: True
Contents: ['wnids.txt', 'val', 'train', 'words.txt', 'test']
Found 200 class folders in /content/tiny-imagenet-200/train. Expected 200 for Tiny ImageNet.


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 124MB/s]


In [ ]:

num_epochs = 20
early_stopping = EarlyStopping(patience=7)

for epoch in range(num_epochs):
   
    model.train()
    running_loss = 0.0
    top1_acc.reset()
    top5_acc.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        top1_acc.update(outputs, labels)
        top5_acc.update(outputs, labels)
        precision.update(outputs, labels)
        recall.update(outputs, labels)
        f1.update(outputs, labels)

    train_loss = running_loss / len(train_loader)
    train_top1 = top1_acc.compute()
    train_top5 = top5_acc.compute()
    train_precision = precision.compute()
    train_recall = recall.compute()
    train_f1 = f1.compute()

    
    model.eval()
    val_loss = 0.0
    top1_acc.reset()
    top5_acc.reset()
    precision.reset()
    recall.reset()
    f1.reset()

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            top1_acc.update(outputs, labels)
            top5_acc.update(outputs, labels)
            precision.update(outputs, labels)
            recall.update(outputs, labels)
            f1.update(outputs, labels)

    val_loss = val_loss / len(val_loader)
    val_top1 = top1_acc.compute()
    val_top5 = top5_acc.compute()
    val_precision = precision.compute()
    val_recall = recall.compute()
    val_f1 = f1.compute()

    
    print(f'Epoch {epoch+1}:')
    print(f'Train - Loss: {train_loss:.4f}, TOP1 Acc: {train_top1:.4f}, TOP5 Acc: {train_top5:.4f}, Precision: {train_precision:.4f}, Recall: {train_recall:.4f}, F1: {train_f1:.4f}')
    print(f'Val   - Loss: {val_loss:.4f}, TOP1 Acc: {val_top1:.4f}, TOP5 Acc: {val_top5:.4f}, Precision: {val_precision:.4f}, Recall: {val_recall:.4f}, F1: {val_f1:.4f}')

    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break

Epoch 1:
Train - Loss: 4.2157, TOP1 Acc: 0.1001, TOP5 Acc: 0.2855, Precision: 0.0835, Recall: 0.1001, F1: 0.0873
Val   - Loss: 3.4327, TOP1 Acc: 0.2065, TOP5 Acc: 0.4726, Precision: 0.2133, Recall: 0.2065, F1: 0.1705
Epoch 2:
Train - Loss: 3.6357, TOP1 Acc: 0.1866, TOP5 Acc: 0.4318, Precision: 0.1670, Recall: 0.1866, F1: 0.1711
Val   - Loss: 3.0673, TOP1 Acc: 0.2698, TOP5 Acc: 0.5645, Precision: 0.2931, Recall: 0.2698, F1: 0.2394
Epoch 3:
Train - Loss: 3.4039, TOP1 Acc: 0.2296, TOP5 Acc: 0.4876, Precision: 0.2123, Recall: 0.2296, F1: 0.2157
Val   - Loss: 2.7878, TOP1 Acc: 0.3321, TOP5 Acc: 0.6262, Precision: 0.3644, Recall: 0.3321, F1: 0.3127
Epoch 4:
Train - Loss: 3.2848, TOP1 Acc: 0.2543, TOP5 Acc: 0.5157, Precision: 0.2387, Recall: 0.2543, F1: 0.2416
Val   - Loss: 2.6770, TOP1 Acc: 0.3587, TOP5 Acc: 0.6483, Precision: 0.3865, Recall: 0.3587, F1: 0.3405
Epoch 5:
Train - Loss: 3.1542, TOP1 Acc: 0.2801, TOP5 Acc: 0.5436, Precision: 0.2655, Recall: 0.2801, F1: 0.2683
Val   - Loss: 2.592